In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "../data/raw/Online Retail.xlsx"
df = pd.read_excel(DATA_PATH)
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

In [5]:
# Bước 1: Khử trùng lặp 100% tất cả các trường
df1 = df.drop_duplicates(keep="first").copy()

# Bước 2: Loại bỏ dòng thiếu CustomerID
df2 = df1.dropna(subset=["CustomerID"]).copy()
df2["CustomerID"] = df2["CustomerID"].astype(int)

# Bước 3: Khử cặp đơn đặt nhầm cực trị khách 16446 (80.995 cái)
df3 = df2[~df2["InvoiceNo"].isin(["581483", "C581484"])].copy()

# Danh sách mã phi hàng hóa thống nhất cho cả hai luồng
FEE_CODES = ["POST", "D", "M", "CRUK", "DOT", "BANK CHARGES", "AMAZONFEE", "B", "C2"]

# Tách luồng hoàn trả thô (trước khi lọc mã phí)
returns_raw = df3[df3["Quantity"] < 0].copy()

# Bước 4: Tách luồng mua hàng (Quantity > 0 & UnitPrice > 0)
df4 = df3[(df3["Quantity"] > 0) & (df3["UnitPrice"] > 0)].copy()

# Bước 5: Loại bỏ mã dịch vụ/phí vận chuyển ở cả hai luồng
purchases_clean = df4[~df4["StockCode"].isin(FEE_CODES)].copy()
returns_clean = returns_raw[~returns_raw["StockCode"].isin(FEE_CODES)].copy()

display([purchases_clean.head(), returns_clean.head()])

[  InvoiceNo StockCode                          Description  Quantity  \
 0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
 1    536365     71053                  WHITE METAL LANTERN         6   
 2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
 3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
 4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
 
           InvoiceDate  UnitPrice  CustomerID         Country  
 0 2010-12-01 08:26:00       2.55       17850  United Kingdom  
 1 2010-12-01 08:26:00       3.39       17850  United Kingdom  
 2 2010-12-01 08:26:00       2.75       17850  United Kingdom  
 3 2010-12-01 08:26:00       3.39       17850  United Kingdom  
 4 2010-12-01 08:26:00       3.39       17850  United Kingdom  ,
     InvoiceNo StockCode                        Description  Quantity  \
 154   C536383    35004C    SET OF 3 COLOURED  FLYING DUCKS        -1   
 235   C536391     22556     

In [10]:
# Đồng nhất thời gian mỗi hóa đơn bằng thời điểm dòng đầu tiên
purchases_clean["InvoiceDate"] = purchases_clean.groupby("InvoiceNo")["InvoiceDate"].transform("min")

# Chuẩn hóa 1 khách hàng chỉ gắn với 1 quốc gia (theo đơn hàng mới nhất)
latest_country = purchases_clean.sort_values("InvoiceDate").groupby("CustomerID")["Country"].last()
purchases_clean["Country"] = purchases_clean["CustomerID"].map(latest_country)

# Tính doanh thu dòng
purchases_clean["TotalAmount"] = (purchases_clean["Quantity"] * purchases_clean["UnitPrice"]).round(4)

# Tháng giao dịch (lưu dưới dạng string YYYY-MM để tương thích hoàn toàn với Parquet)
purchases_clean["InvoiceMonth"] = purchases_clean["InvoiceDate"].dt.to_period("M").astype(str)

# Tháng mua hàng đầu tiên của mỗi khách hàng (Cohort Month)
purchases_clean["CohortMonth"] = purchases_clean.groupby("CustomerID")["InvoiceMonth"].transform("min")

purchases_clean

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,InvoiceMonth,CohortMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,2010-12
...,...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20,2011-12,2011-08
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,2011-12,2011-08
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,2011-12,2011-08
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,2011-12,2011-08


In [11]:
# Kiểm tra khớp thứ tự thời gian trên 8.505 dòng hàng hóa hoàn trả bằng vector hóa tốc độ cao
min_p_date = purchases_clean.groupby(["CustomerID", "StockCode"])["InvoiceDate"].min()
returns_clean["MinPurchaseDate"] = returns_clean.set_index(["CustomerID", "StockCode"]).index.map(min_p_date)
returns_clean["IsMatched"] = returns_clean["MinPurchaseDate"].notna() & (returns_clean["MinPurchaseDate"] <= returns_clean["InvoiceDate"])
returns_clean["ReturnAmount"] = returns_clean["Quantity"] * returns_clean["UnitPrice"]


n_matched = returns_clean["IsMatched"].sum()
val_matched = returns_clean[returns_clean["IsMatched"]]["ReturnAmount"].sum()

n_orphan = (~returns_clean["IsMatched"]).sum()
val_orphan = returns_clean[~returns_clean["IsMatched"]]["ReturnAmount"].sum()

print(f"Đơn hủy có đơn mua gốc trước đó: {n_matched:,} dòng ({n_matched/len(returns_clean):.1%}) | Giá trị: £{val_matched:,.2f}")
print(f"Đơn hủy mồ côi (trước 12/2010): {n_orphan:,} dòng ({n_orphan/len(returns_clean):.1%}) | Giá trị: £{val_orphan:,.2f} ({abs(val_orphan)/abs(returns_clean['ReturnAmount'].sum()):.1%})")

# Tính Matched Net Spend cho từng khách hàng
cust_gross = purchases_clean.groupby("CustomerID")["TotalAmount"].sum()
matched_returns = returns_clean[returns_clean["IsMatched"]].groupby("CustomerID")["ReturnAmount"].sum()
cust_matched_net = cust_gross.add(matched_returns, fill_value=0)

all_returns = returns_clean.groupby("CustomerID")["ReturnAmount"].sum()
cust_all_net = cust_gross.add(all_returns, fill_value=0)

print(f"Số khách có Matched Net Spend <= 0: {(cust_matched_net <= 0).sum()} khách")
print(f"Số khách có All Net Spend <= 0 (do đơn mồ côi): {(cust_all_net <= 0).sum()} khách (chênh lệch đúng 32 khách do cắt cụt thời gian)")

Đơn hủy có đơn mua gốc trước đó: 7,477 dòng (87.9%) | Giá trị: £-281,114.92
Đơn hủy mồ côi (trước 12/2010): 1,028 dòng (12.1%) | Giá trị: £-22,166.19 (7.3%)
Số khách có Matched Net Spend <= 0: 7 khách
Số khách có All Net Spend <= 0 (do đơn mồ côi): 39 khách (chênh lệch đúng 32 khách do cắt cụt thời gian)


In [ ]:
# Xuất dữ liệu ra data/processed/
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

returns_export = returns_clean.drop(columns=["MinPurchaseDate"], errors="ignore").copy()

purchases_clean.to_parquet(f"{OUTPUT_DIR}/online_retail_clean.parquet", index=False)
returns_export.to_parquet(f"{OUTPUT_DIR}/returns_clean.parquet", index=False)

purchases_clean.to_csv(f"{OUTPUT_DIR}/online_retail_clean.csv", index=False)
returns_export.to_csv(f"{OUTPUT_DIR}/returns_clean.csv", index=False)

>>> TẤT CẢ CÁC TIÊU CHÍ NGHIỆM THU ĐÃ ĐẠT 100%! <<<
Đã lưu thành công cả 2 bảng Parquet và CSV vào: ../data/processed/
